In [17]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table   
from astropy.coordinates import SkyCoord     
from astropy import units as u                   
from astropy.cosmology import Planck18 as cosmo

In [ ]:
#read in the catalog
path_to_catalog = '/fs/ess/PCON0003/desi-visualizations/catalogs/zall-tilecumulative-fuji.fits'  
catalog = Table.read(path_to_catalog)

In [ ]:
# trim catalog
RA_max = 360   # RA ranges from 0-360
DEC_max = 90   # DEC ranges from -90 to 90 
catalog = catalog[(catalog['TARGET_RA'] < RA_max) & (catalog['TARGET_DEC'] < DEC_max)]

# limit redshift
z_min = 0.01 # below this, they're mostly stars or errors.
z_max = 3    # DESI goes higher than this, but it is sparse and harder to visualize.
catalog = catalog[(catalog['Z'] > z_min) & (catalog['Z'] < z_max)]

# remove galaxies with unrealisting imaging fluxes
flux_g_max = 50  # these are quite faint. You calso can't have negative flux!
flux_g_min = 0
flux_r_max = 50  
flux_r_min = 0
flux_z_max = 50  
flux_z_min = 0
flux_W1_max = 50  
flux_W1_min = 0
flux_W2_max = 50  
flux_W2_min = 0
catalog = catalog[(catalog['FLUX_G'] < flux_g_max) & (catalog['FLUX_G'] > flux_g_min) & (catalog['FLUX_R'] < flux_r_max) & (catalog['FLUX_R'] > flux_r_min) & (catalog['FLUX_Z'] < flux_z_max) & (catalog['FLUX_Z'] > flux_z_min) & (catalog['FLUX_W1'] < flux_W1_max) & (catalog['FLUX_W1'] > flux_W1_min) & (catalog['FLUX_W2'] < flux_W2_max) & (catalog['FLUX_W2'] > flux_W2_min)]

In [20]:
comoving_distance = cosmo.comoving_distance(catalog['Z'])  # calculate the comoving distance to each object based on its redshift and the cosmology (to know: what is comoving distance?)
coords = SkyCoord(ra=catalog['TARGET_RA']*u.degree, dec=catalog['TARGET_DEC']*u.degree, distance=comoving_distance)  # astropy converts this into a 3D coordinate system

In [21]:
#define 3d coords
x = coords.cartesian.x  # these are still astropy objects, i.e. they have assigned units (Mpc)
y = coords.cartesian.y
z = coords.cartesian.z

In [22]:
#define redshift and fluxes
redshift = catalog['Z'] 
flux_g = catalog['FLUX_G']
flux_r = catalog['FLUX_R']
flux_z = catalog['FLUX_Z']
flux_w1 = catalog['FLUX_W1']
flux_w2 = catalog['FLUX_W2']
#xtra
object_type = catalog['SPECTYPE']
zerr = catalog['ZERR']
zwarn = catalog['ZWARN']

In [23]:
#define apparent magnitudes
app_mag_G = -2.5*np.log10(catalog['FLUX_G'])+22.5 
app_mag_R = -2.5*np.log10(catalog['FLUX_R'])+22.5 
app_mag_Z = -2.5*np.log10(catalog['FLUX_Z'])+22.5 
app_mag_W1 = -2.5*np.log10(catalog['FLUX_W1'])+22.5
app_mag_W2 = -2.5*np.log10(catalog['FLUX_W2'])+22.5


In [24]:
#define absolute magnitudes
def apparent_to_absolute_magnitude(m, z):
    d_L = cosmo.luminosity_distance(z)  
    M = m - 5 * np.log10(d_L.to(u.pc).value) + 5  
    return M
abs_mag_g = apparent_to_absolute_magnitude(app_mag_G, redshift)
abs_mag_r = apparent_to_absolute_magnitude(app_mag_R, redshift)
abs_mag_z = apparent_to_absolute_magnitude(app_mag_Z, redshift)
abs_mag_w1 = apparent_to_absolute_magnitude(app_mag_W1, redshift)
abs_mag_w2 = apparent_to_absolute_magnitude(app_mag_W2, redshift)


In [26]:
table = Table([x, y, z, redshift, zerr, zwarn, object_type, flux_g, flux_r, flux_z, flux_w1, flux_w2, app_mag_G, app_mag_R, app_mag_Z, app_mag_W1, app_mag_W2, abs_mag_g, abs_mag_r, abs_mag_z, abs_mag_w1, abs_mag_w2], names=['x', 'y', 'z', 'redshift', 'zerr', 'zwarn', 'object_type', 'flux_g', 'flux_r', 'flux_z', 'flux_w1', 'flux_w2', 'app_mag_G', 'app_mag_R', 'app_mag_Z', 'app_mag_W1', 'app_mag_W2', 'abs_mag_g', 'abs_mag_r', 'abs_mag_z', 'abs_mag_w1', 'abs_mag_w2'])
table.write('desi_catalog.csv', format='csv', overwrite=True)